# 编写自定义tokenizer

In [11]:
# pip install re
import re

In [12]:

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of character:", len(raw_text))
print(raw_text[:99])


Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [13]:
# 这里是把原始文本用正则分割成词或标记
tokens = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
tokens = [t.strip() for t in tokens if t.strip()]  # 去除空白和空字符串
print(tokens[:30])  # 打印前30个词或标记

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [16]:
# 这里的 preprocessed 未定义，会报错。实际上，我们希望 all_words 能获得所有不重复的token。
# 应该用上面已经分好词的 tokens 作为“预处理后”的数据。
all_words = sorted(set(tokens))
vocab_size = len(all_words)

print(vocab_size)  # 词表大小

1130


In [17]:
vocab = {token: integer for integer, token in enumerate(all_words)}

In [18]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [19]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [t.strip() for t in tokens if t.strip()]
        ids = [self.str_to_int[s] for s in tokens]
        return ids

    def decode(self, ids):
        words = [self.int_to_str[i] for i in ids]
        text = " ".join(words)
        # 去除某些标点前的空格
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
# 实例化一个SimpleTokenizerV1对象，传入词表vocab
tokenizer = SimpleTokenizerV1(vocab)

# 需要被编码的原始文本
text =""""It's the last he painted, you know,"Mrs. Gisburn said with pardonable pride."""
# 使用tokenizer对文本进行编码，得到id序列
ids = tokenizer.encode(text)
# 打印编码后的id序列
print(ids)#

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [ ]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [ ]:
text = "Hello, do you like tea?"
print(tokenizer.encode(text))

KeyError: 'Hello'

In [ ]:
all_tokens = sorted(set(preprocessed))
all_tokens += ["<|endoftext|>", "<|unk|>"]

vocab = {token: idx for idx, token in enumerate(all_tokens)}

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        # 为什么这里能把hello添加进去？
        # 因为在实例化SimpleTokenizerV2时传入的vocab已经包含了"hello"（全部词表都是从语料preprocessed统计得到的，并且后面又补充了<|endoftext|>和<|unk|>，如cell 11），所以无论输入text中出现什么（如"Hello"），如果它在vocab里就能被正常编码；
        # 如果不在，就会被替换成"<|unk|>"编码
        tokens = re.split(r'([,.:;?_!"()\']|--|\s+)', text)
        tokens = [tok for tok in tokens if tok and not tok.isspace()]
        tokens = [tok if tok in self.str_to_int else "<|unk|>" for tok in tokens]
        ids = [self.str_to_int[tok] for tok in tokens]
        return ids

    def decode(self, ids):
        tokens = [self.int_to_str[i] for i in ids]
        text = ""
        for idx, token in enumerate(tokens):
            if idx > 0 and not re.match(r'[,.:;?_!"()\']|--', token):
                text += " "
            text += token
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text ="Hello, do you like tea?"
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10]


# BPE(Byte pair encoding)

In [ ]:
pip install tiktoken

   ---------------------------------------- 0.0/878.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/878.7 kB ? eta -:--:--
   ----------------------- ---------------- 524.3/878.7 kB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 878.7/878.7 kB 4.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces" "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [ ]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


输出的文本 strings 有一个细微的区别：sunlit terracesof someunknownPlace 中缺少了空格。这个问题出现在词汇表的细粒度拆分上，子词分割可能将 terraces 和 of 合并成了一个词，因此没有在这两个部分之间添加空格。BPE或其他子词分割方法可能将某些词合并成子词序列，这会导致在解码时出现一些合并现象。

# 使用BPE为GPT自回归预训练阶段准备数据集

## 导入数据库

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_sample = tokenizer.encode(raw_text)
print(len(enc_sample))  # 5145

4690


In [ ]:
context_size = 4

In [ ]:
# Check the shape and content of x and y
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print("x:", x)
print("y:", y)


x: [53, 44, 149, 1003]
y: [44, 149, 1003, 57]


In [ ]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[53] ----> 44
[53, 44] ----> 149
[53, 44, 149] ----> 1003
[53, 44, 149, 1003] ----> 57


In [ ]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

I ----> HAD
I HAD ----> always
I HAD always ----> thought
I HAD always thought ----> Jack


In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
import tiktoken

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # max_length is context size
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=4,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
# 创建数据加载器
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

In [ ]:
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


# BPE迭代可视化

In [ ]:
# =====================================
# 简易 BPE 可视化：打印每次 merge 的表格
# =====================================

from collections import Counter, defaultdict

# 1. 玩具语料（你可以改成自己的）
#   这里刻意选一些带 est/er 的词，方便看到类似 “e + s -> es”, "es + t -> est" 的过程
corpus = [
    "lower",
    "lowest",
    "newer",
    "wider",
    "finest",
    "older",
]

# 每个词拆成字符 + </w> 结尾
def make_vocab(words):
    vocab = {}
    for w in words:
        tokens = list(w) + ["</w>"]
        vocab[tuple(tokens)] = vocab.get(tuple(tokens), 0) + 1
    return vocab

vocab = make_vocab(corpus)


def get_symbol_freq(vocab):
    """统计每个 token 在整个语料中的总出现频率"""
    freq = Counter()
    for word, count in vocab.items():
        for sym in word:
            freq[sym] += count
    return freq


def get_pair_freq(vocab):
    """统计每个相邻 pair 的出现频率"""
    pair_freq = defaultdict(int)
    for word, count in vocab.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i+1])
            pair_freq[pair] += count
    return pair_freq


def merge_pair(vocab, merge_pair):
    """在所有词中把指定的 pair 合并成一个新 token"""
    a, b = merge_pair
    new_vocab = {}
    new_token = a + b  # 简单拼接，比如 'e' + 's' -> 'es'

    for word, count in vocab.items():
        new_word = []
        i = 0
        while i < len(word):
            # 如果当前位置和下一个位置刚好是要 merge 的 pair
            if i < len(word) - 1 and word[i] == a and word[i+1] == b:
                new_word.append(new_token)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_vocab[tuple(new_word)] = new_vocab.get(tuple(new_word), 0) + count

    return new_vocab, new_token


def print_symbol_table(symbol_freq, iteration):
    """按频率排序打印当前 token 表"""
    print(f"\n===== Iteration {iteration}：当前 token 频率表 =====")
    print(f"{'Number':>6}  {'Token':<10}  {'Frequency':>9}")
    print("-" * 32)
    for idx, (sym, freq) in enumerate(symbol_freq.most_common(), start=1):
        print(f"{idx:>6}  {sym:<10}  {freq:>9}")
    print("-" * 32)


def run_bpe(vocab, num_merges=10):
    for it in range(1, num_merges + 1):
        # 1. 打印当前 token 频率
        symbol_freq = get_symbol_freq(vocab)
        print_symbol_table(symbol_freq, it)

        # 2. 找到出现次数最多的相邻 pair
        pair_freq = get_pair_freq(vocab)
        if not pair_freq:
            print("没有可合并的 pair 了，停止。")
            break
        best_pair, best_count = max(pair_freq.items(), key=lambda x: x[1])

        print(f"本轮选择合并的 pair: {best_pair[0]} + {best_pair[1]}  ->  {best_pair[0] + best_pair[1]} "
              f"(出现 {best_count} 次)")

        # 3. 执行合并，得到新的 vocab
        vocab, new_token = merge_pair(vocab, best_pair)

    # 最后打印一次最终 vocab 中的单词形式
    print("\n===== 最终分词结果（每个词的 token 序列） =====")
    for word, count in vocab.items():
        print(f"{''.join([t.replace('</w>', '') for t in word])!r:>10}  ->  {word} (freq={count})")


if __name__ == "__main__":
    print("原始语料：", corpus)
    run_bpe(vocab, num_merges=5)  # 你可以调大/调小 merge 次数


原始语料： ['lower', 'lowest', 'newer', 'wider', 'finest', 'older']

===== Iteration 1：当前 token 频率表 =====
Number  Token       Frequency
--------------------------------
     1  e                   7
     2  </w>                6
     3  w                   4
     4  r                   4
     5  l                   3
     6  o                   3
     7  s                   2
     8  t                   2
     9  n                   2
    10  i                   2
    11  d                   2
    12  f                   1
--------------------------------
本轮选择合并的 pair: e + r  ->  er (出现 4 次)

===== Iteration 2：当前 token 频率表 =====
Number  Token       Frequency
--------------------------------
     1  </w>                6
     2  w                   4
     3  er                  4
     4  l                   3
     5  o                   3
     6  e                   3
     7  s                   2
     8  t                   2
     9  n                   2
    10  i                   2
    1

尽可能在有限词表大小里合并最有价值的高频子词。

## 原始语料： ['lower', 'lowest', 'newer', 'wider', 'finest', 'older']
--------------------------------
Iteration 1：当前 token 频率表
Number  Token       Frequency
--------------------------------
     1  e                   7
     2  </w>                6
     3  w                   4
     4  r                   4
     5  l                   3
     6  o                   3
     7  s                   2
     8  t                   2
     9  n                   2
    10  i                   2
    11  d                   2
    12  f                   1
--------------------------------
本轮选择合并的 pair: e + r  ->  er (出现 4 次)

Iteration 2：当前 token 频率表 
Number  Token       Frequency
--------------------------------
     1  </w>                6
     2  w                   4
     3  er                  4
     4  l                   3
     5  o                   3
     6  e                   3
     7  s                   2
     8  t                   2
     9  n                   2
    10  i                   2
    11  d                   2
    12  f                   1
--------------------------------
本轮选择合并的 pair: er + </w>  ->  er</w> (出现 4 次)

Iteration 3：当前 token 频率表 
Number  Token       Frequency
--------------------------------
     1  w                   4
     2  er</w>              4
     3  l                   3
     4  o                   3
     5  e                   3
     6  s                   2
     7  t                   2
     8  </w>                2
     9  n                   2
    10  i                   2
    11  d                   2
    12  f                   1
--------------------------------
本轮选择合并的 pair: l + o  ->  lo (出现 2 次)

Iteration 4：当前 token 频率表
Number  Token       Frequency
--------------------------------
     1  w                   4
     2  er</w>              4
     3  e                   3
     4  lo                  2
     5  s                   2
     6  t                   2
     7  </w>                2
     8  n                   2
     9  i                   2
    10  d                   2
    11  f                   1
    12  o                   1
    13  l                   1
--------------------------------
本轮选择合并的 pair: lo + w  ->  low (出现 2 次)

Iteration 5：当前 token 频率表 
Number  Token       Frequency
--------------------------------
     1  er</w>              4
     2  e                   3
     3  low                 2
     4  s                   2
     5  t                   2
     6  </w>                2
     7  n                   2
     8  w                   2
     9  i                   2
    10  d                   2
    11  f                   1
    12  o                   1
    13  l                   1
--------------------------------
本轮选择合并的 pair: e + s  ->  es (出现 2 次)

### | 词语      | Token 序列                        | 频率 (freq) |
### |-----------|-----------------------------------|-------------|
### | lower     | ('low', 'er</w>')                | 1           |
### | lowest    | ('low', 'es', 't', '</w>')       | 1           |
### | newer     | ('n', 'e', 'w', 'er</w>')        | 1           |
### | wider     | ('w', 'i', 'd', 'er</w>')        | 1           |
### | finest    | ('f', 'i', 'n', 'es', 't', '</w>')| 1           |
### | older     | ('o', 'l', 'd', 'er</w>')        | 1           |